# Milestone 1 — Colab CPU
Notebook smoke tối đa 10 ID; code lấy từ GitHub, dữ liệu lưu trong runtime/ dưới Drive root bạn chọn. Sau khi cập nhật code, restart runtime rồi chạy lại từ bước mount/root để tránh module Python cũ còn trong RAM.


In [ ]:
from pathlib import Path
import os, subprocess, sys
selected = Path(input('Thư mục dự án đã chọn trên Google Drive: ')).expanduser()
assert selected.is_dir(), 'Chọn thư mục Drive đã mount và tồn tại'
assert input('Xác nhận thư mục này nằm trên Drive (yes): ') == 'yes'
os.chdir(selected)
print('Project root selected')

In [ ]:
repo = Path('code')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/NewB-S2deeC0de/arXiv.git',str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(repo/'src/requirements.txt')],check=True)
sys.path.insert(0,str(repo/'src'))

In [ ]:
from crawler.config import load_config
from crawler.pipeline import run_pipeline
from crawler.secrets import load_semantic_scholar_key
config = load_config(Path('code/config/crawler.yaml'))
print('authentication=' + ('key' if load_semantic_scholar_key() else 'anonymous'))

## Smoke: tối đa 10 ID
Cell này gọi mạng thật khi bạn chủ động chạy. Dữ liệu nằm trong runtime/ tương đối với Drive root đã chọn.

In [ ]:
summary = await run_pipeline(config, {'arxiv','s2'}, resume=True, limit=10)
print(summary)

## Status và resume
Ngắt cell chạy bằng nút Stop, sau đó chạy lại cell smoke. Chỉ đặt takeover_stale=True nếu runtime trước đã dừng hoàn toàn.

In [ ]:
from crawler.progress import ProgressStore
store = ProgressStore(config.paths['persistent_root'])
print(store.status_counts())


## Khi nhận API key
Thêm SEMANTIC_SCHOLAR_API_KEY trong Colab Secrets, bật quyền notebook. Chạy lại cell kiểm tra authentication, rồi cell smoke. Các stage failed chỉ được thử lại khi chủ động chạy cell sau; không reset dữ liệu thành công.

In [ ]:
from crawler.diagnostics import diagnose_sources
import json
print(json.dumps(diagnose_sources(config), indent=2, ensure_ascii=False))


In [ ]:
# Chỉ bật sau khi đã xem diagnosis và chắc chắn không có phiên khác đang chạy.
RETRY_FAILED_SOURCES = False
if RETRY_FAILED_SOURCES:
    from crawler.locking import DatasetLock
    from crawler.ids import generate_ids
    import uuid
    with DatasetLock(config.paths['persistent_root'], 'retry-' + uuid.uuid4().hex):
        store = ProgressStore(config.paths['persistent_root'])
        store.reset('failed', stages={'source'}, ids=set(generate_ids(**config.range)[:10]))
    summary = await run_pipeline(config, {'arxiv'}, resume=True, limit=10)
    print(summary)


In [ ]:
from crawler.export import export_snapshot
import uuid
manifest = export_snapshot(config, 'export-' + uuid.uuid4().hex)
print(manifest)


## Sau smoke và interrupt/resume
CLI không truyền --limit sẽ chọn toàn range; notebook này luôn dùng limit=10. Chỉ chạy toàn bộ sau khi bạn kiểm chứng smoke mới và ngắt/resume trên Drive. Không chạy hai phiên cùng dataset.
